# Earthquakes Analysis with Elasticsearch

Dans ce notebook, nous allons interroger les données de tremblements de terre et de blasts
ingérées dans Elasticsearch par le conteneur `ingest-init`.

Indices utilisés :
- `ncedc-earthquakes-earthquake`
- `ncedc-earthquakes-blast`

Assurez-vous que :
- `docker compose up -d` a été lancé
- le job d'ingestion a terminé (voir logs du conteneur `ingest-init`)

## Mettre en place la connexion à Elasticsearch

In [ ]:
from elasticsearch import Elasticsearch

ES_HOST = "http://elasticsearch:9200"
ES_USER = "elastic"
ES_PASSWORD = "111"  # ou récupérer via variable d'env si tu préfères

es = Elasticsearch(
    ES_HOST,
    basic_auth=(ES_USER, ES_PASSWORD),
)

if es.ping():
    print("Connected to Elasticsearch!")
else:
    raise RuntimeError("Could not connect to Elasticsearch")


## Vérifier que les données sont présentes dans les indices

In [ ]:
indices = ["ncedc-earthquakes-earthquake", "ncedc-earthquakes-blast"]

for index in indices:
    if not es.indices.exists(index=index):
        es.indices.create(index=index)
        print(f"Created empty index: {index}")
    else:
        print(f"Index already exists: {index}")

Combien de documents sont présents dans chaque indice ?

In [ ]:
for index in indices:
    if es.indices.exists(index=index):
        count = es.count(index=index)["count"]
        print(f"{index}: {count} documents")
    else:
        print(f"{index} does NOT exist")

Quelles sont les sources des documents présents dans les indices ?

In [ ]:
resp = es.search(
    index="*earthquakes*",
    size=5,
    query={"match_all": {}},
)
for hit in resp["hits"]["hits"]:
    print(hit["_source"])


Que fait cette cellule ?

In [ ]:
body = {
    "size": 0,
    "aggs": {
        "mag_hist": {
            "histogram": {
                "field": "mag",
                "interval": 0.5,
                "min_doc_count": 1,
            }
        }
    },
}

resp_eq = es.search(index="ncedc-earthquakes-earthquake", body=body)
buckets = resp_eq["aggregations"]["mag_hist"]["buckets"]
buckets[:5]

<details Solution>
  <summary>Solution</summary>
  Cette cellule crée un histogramme des magnitudes des tremblements de terre :

- body — Définit une requête Elasticsearch avec :
- "size": 0 → ne pas récupérer les documents individuels, seulement les agrégations
- "aggs" → créer une agrégation nommée mag_hist
- "histogram" → regrouper par intervalles (buckets)
- "field": "mag" → sur le champ magnitude
- "interval": 0.5 → chaque bucket couvre 0.5 magnitude (ex: 0-0.5, 0.5-1.0, etc.)
- "min_doc_count": 1 → ignorer les buckets vides
- es.search(...) — Exécute la requête sur l'indice ncedc-earthquakes-earthquake

- buckets — Extrait les résultats (les groupes de magnitudes avec leurs comptages)

- buckets[:5] — Affiche les 5 premiers buckets

Résultat → Un tableau montrant combien de tremblements de terre existent pour chaque plage de magnitude (ex: 5 séismes de mag 1.0-1.5, 12 séismes de mag 1.5-2.0, etc.)
</details>   

## Distribution des magnitudes

On commence par un histogramme de la magnitude (`mag`) pour les tremblements de terre.
Chaque bucket représente un intervalle de 0.5 de magnitude.

In [ ]:
def mag_hist(index_name: str):
    body = {
        "size": 0,
        "aggs": {
            "mag_hist": {
                "histogram": {
                    "field": "mag",
                    "interval": 0.5,
                    "min_doc_count": 1,
                }
            }
        },
    }
    resp = es.search(index=index_name, body=body)
    return resp["aggregations"]["mag_hist"]["buckets"]

eq_buckets = mag_hist("ncedc-earthquakes-earthquake")
blast_buckets = mag_hist("ncedc-earthquakes-blast")


In [ ]:
body = {
    "size": 0,
    "aggs": {
        "per_month": {
            "date_histogram": {
                "field": "@timestamp",
                "calendar_interval": "1M",
                "min_doc_count": 1,
            }
        }
    },
}

resp_eq_time = es.search(index="ncedc-earthquakes-earthquake", body=body)
time_buckets = resp_eq_time["aggregations"]["per_month"]["buckets"]
time_buckets[:5]


## Évolution du nombre de tremblements de terre dans le temps

On regroupe les événements par mois en utilisant un `date_histogram` sur `@timestamp`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def buckets_to_df(buckets, label):
    return pd.DataFrame(
        {
            "mag": [b["key"] for b in buckets],
            "count": [b["doc_count"] for b in buckets],
            "type": label,
        }
    )

df_eq = buckets_to_df(eq_buckets, "earthquake")
df_blast = buckets_to_df(blast_buckets, "blast")

df_mag = pd.concat([df_eq, df_blast], ignore_index=True)
df_mag.head()


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_mag,
    x="mag",
    y="count",
    hue="type",
)
plt.title("Distribution des magnitudes : earthquakes vs blasts")
plt.xlabel("Magnitude (bins de 0.5)")
plt.ylabel("Nombre d'événements")
plt.tight_layout()
plt.show()


In [ ]:
df_time = pd.DataFrame(
    {
        "date": [pd.to_datetime(b["key_as_string"]) for b in time_buckets],
        "count": [b["doc_count"] for b in time_buckets],
    }
)

plt.figure(figsize=(12, 4))
plt.plot(df_time["date"], df_time["count"])
plt.title("Nombre de tremblements de terre par mois")
plt.xlabel("Date")
plt.ylabel("Nombre d'événements")
plt.tight_layout()
plt.show()